2 silver.douyin_aweme_hashtag

Each row = 1 hashtag of 1 aweme.

Field:
aweme_id
account_id
hashtag_name
hashtag_id
hashtag_type
is_commerce
caption_start
caption_end

In [0]:
from pyspark.sql import DataFrame, functions as F
from pyspark.sql.window import Window


bronze_table = "de_e2e.bronze.douyin_api_raw"
silver_table = "de_e2e.silver.douyin_aweme_hashtag"

bucket = "de-e2e-413612133697-ap-southeast-1-an"
silver_path = f"s3://{bucket}/lakehouse/silver/douyin/aweme_hashtag_delta/"

In [0]:
bronze_df = spark.read.table(bronze_table)

#Current struct in S3: raw_struct.raw.raw.aweme_list
def with_aweme_array(df: DataFrame) -> DataFrame:
    return df.withColumn(
        "aweme_list",
        F.col("raw_struct.raw.raw.aweme_list"),
    )

# Turn user video list to flat rows, preserving account_id even if aweme_list is null/empty
hashtag_df = (
    bronze_df
    .transform(with_aweme_array)
    .withColumn("aweme", F.explode_outer("aweme_list"))
    .withColumn("hashtag", F.explode_outer("aweme.text_extra"))
    .select(
        F.col("niche"),
        F.col("account_id"),
        F.col("source_file"),
        F.col("generated_at").alias("landing_generated_at"),
        F.col("bronze_ingested_at"),

        F.col("aweme.aweme_id").cast("string").alias("aweme_id"),
        F.col("aweme.group_id").cast("string").alias("group_id"),
        F.col("aweme.sec_item_id").cast("string").alias("sec_item_id"),
        F.col("aweme.desc").alias("description"),
        F.col("aweme.create_time").cast("long").alias("create_time_epoch"),
        F.from_unixtime(F.col("aweme.create_time").cast("long")).cast("timestamp").alias("created_at"),
        F.col("aweme.author.uid").cast("string").alias("author_uid"),
        F.col("aweme.author.nickname").alias("author_nickname"),

        F.col("hashtag.hashtag_name").alias("hashtag_name"),
        F.col("hashtag.hashtag_id").cast("string").alias("hashtag_id"),
        F.col("hashtag.type").cast("int").alias("hashtag_type"),
        F.col("hashtag.is_commerce").cast("boolean").alias("is_commerce"),
        F.col("hashtag.start").cast("int").alias("text_start"),
        F.col("hashtag.end").cast("int").alias("text_end"),
        F.col("hashtag.caption_start").cast("int").alias("caption_start"),
        F.col("hashtag.caption_end").cast("int").alias("caption_end"),
        F.col("hashtag").alias("raw_hashtag"),

        F.current_timestamp().alias("silver_updated_at"),
    )
    .where(F.col("aweme_id").isNotNull())
    .where(F.col("hashtag_name").isNotNull())
)

window_spec = Window.partitionBy("account_id", "aweme_id", "hashtag_name").orderBy(
    F.col("landing_generated_at").desc_nulls_last(),
    F.col("bronze_ingested_at").desc_nulls_last(),
    F.col("source_file").desc_nulls_last(),
)

hashtag_clean_df = (
    hashtag_df
    .withColumn("rn", F.row_number().over(window_spec))
    .where(F.col("rn") == 1)
    .drop("rn")
)

display(
    hashtag_clean_df.select(
        "account_id",
        "aweme_id",
        "hashtag_name",
        "hashtag_id",
        "description",
        "created_at",
        "author_nickname",
    ).orderBy("account_id", "aweme_id", "hashtag_name").limit(100)
)

In [0]:
from delta.tables import DeltaTable
if DeltaTable.isDeltaTable(spark, silver_path):
    target = DeltaTable.forPath(spark, silver_path)
    (
        target.alias("t")
        .merge(
            hashtag_clean_df.alias("s"),
            """
            t.account_id = s.account_id
            AND t.aweme_id = s.aweme_id
            AND t.hashtag_name = s.hashtag_name
            """,
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )
else:
    (
        hashtag_clean_df.write
        .format("delta")
        .mode("overwrite")
        .save(silver_path)
    )

spark.sql(
    f"""
    CREATE TABLE IF NOT EXISTS {silver_table}
    USING DELTA
    LOCATION '{silver_path}'
    """
)

In [0]:
result_df = spark.table(silver_table)

print(f"Silver hashtag rows: {result_df.count()}")

display(
    result_df.groupBy("hashtag_name")
    .agg(
        F.countDistinct("aweme_id").alias("aweme_count"),
        F.countDistinct("account_id").alias("creator_count"),
        F.max("created_at").alias("latest_created_at"),
    )
    .orderBy(F.col("aweme_count").desc(), F.col("creator_count").desc())
    .limit(50)
)